In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
from tqdm import tqdm, trange
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn import metrics
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler

import lightgbm as lgb

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objs as go
import plotly
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
import cufflinks as cf
cf.set_config_file(offline=True)
# Input data files are available i

import datetime

import warnings
warnings.filterwarnings('ignore')

In [2]:
train_features = pd.read_pickle('data/preprocessed/local_train_features.pkl')
test_features = pd.read_pickle('data/preprocessed/local_test_features.pkl')

In [16]:
for col in train_features.columns:
    print(f"{col}: max={max(train_features[col])}, min={min(train_features[col])}, dtype={train_features[col].dtype}, unique_count={train_features[col].unique().shape[0]}")

building_id: max=1446, min=2, dtype=int16, unique_count=724
meter: max=3, min=0, dtype=int8, unique_count=4
timestamp: max=8783.0, min=0.0, dtype=float64, unique_count=8784
meter_reading: max=274994.0, min=0.0, dtype=float32, unique_count=1185793
site_id: max=15, min=0, dtype=int8, unique_count=16
primary_use: max=Warehouse/storage, min=Education, dtype=category, unique_count=15
square_feet: max=875000, min=283, dtype=int32, unique_count=709
year_built: max=117.0, min=0.0, dtype=float16, unique_count=99
floor_count: max=nan, min=nan, dtype=float16, unique_count=18
air_temperature: max=47.1875, min=-41.5, dtype=float16, unique_count=1240
cloud_coverage: max=126.875, min=-37.40625, dtype=float16, unique_count=13661
dew_temperature: max=26.09375, min=-35.0, dtype=float16, unique_count=1170
precip_depth_1_hr: max=3286.0, min=-12624.0, dtype=float16, unique_count=12022
sea_level_pressure: max=10672.0, min=968.0, dtype=float16, unique_count=302
wind_direction: max=3529.322998046875, min=-171

In [4]:
print(train_features.shape)
print(test_features.shape)

(9688699, 49)
(10527401, 49)


In [5]:
list_variables = list(train_features.drop(['is_bad_meter_reading',
                                  'wind_direction',
                                  'air_temperature_std_lag73'],axis=1).columns)

In [6]:
X_all = train_features[list_variables]
Y_all = train_features['is_bad_meter_reading']

In [7]:
#Data split for taining and validation data
X_train = X_all[X_all['building_id']%5<4]
X_val = X_all[X_all['building_id']%5==4]
Y_train = Y_all[X_all['building_id']%5<4]
Y_val = Y_all[X_all['building_id']%5==4]

print(X_train.shape, X_val.shape)

(7813361, 46) (1875338, 46)


In [8]:
#Normalization
num_cols = X_train.select_dtypes(include=[np.number]).columns
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])
X_test = test_features[list_variables]
X_test[num_cols] = scaler.transform(test_features[num_cols])

In [9]:
#XGBoost modeling
xgb_model = XGBClassifier(
    n_estimators=100,
    tree_method='hist',        # 處理 category 必備的直方圖演算法
    enable_categorical=True,    # 允許模型直接讀取 category 欄位
    random_state=42             # 固定隨機種子，確保每次跑的結果一致
)
xgb_model.fit(X_train, Y_train)

pred_train_xgb = xgb_model.predict_proba(X_train)[:,1]
pred_val_xgb = xgb_model.predict_proba(X_val)[:,1]

score_train_xgb = metrics.roc_auc_score(Y_train, pred_train_xgb)
score_val_xgb = metrics.roc_auc_score(Y_val, pred_val_xgb)

print('Training Accuracy : ', score_train_xgb)
print('Validation Accuracy : ', score_val_xgb)

Training Accuracy :  0.9998693650775883
Validation Accuracy :  0.9248159129899116


In [17]:
#LightGBM modeling
lgb_model = lgb.LGBMClassifier(n_estimators=100)
lgb_model.fit(X_train, Y_train)

pred_train_lgb = lgb_model.predict_proba(X_train)[:,1]
pred_val_lgb = lgb_model.predict_proba(X_val)[:,1]

score_train_lgb = metrics.roc_auc_score(Y_train, pred_train_lgb)
score_val_lgb = metrics.roc_auc_score(Y_val, pred_val_lgb)

print('Training Accuracy : ', score_train_lgb)
print('Validation Accuracy : ', score_val_lgb)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 530542, number of negative: 7282819
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.379435 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 66238
[LightGBM] [Info] Number of data points in the train set: 7813361, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.067902 -> initscore=-2.619374
[LightGBM] [Info] Start training from score -2.619374
Training Accuracy :  0.999464978806577
Validation Accuracy :  0.9425606263292745


In [19]:
#target encoding
from sklearn.preprocessing import TargetEncoder
high_cardinality_cols = [col for col in X_train.columns if X_train[col].nunique() > 255 and X_train[col].dtype == 'category']
print("High cardinality columns:", high_cardinality_cols)

encoder = TargetEncoder(smooth="auto", random_state=42)

X_train_encoded = X_train.copy()
X_val_encoded = X_val.copy()

X_train_encoded[high_cardinality_cols] = encoder.fit_transform(X_train[high_cardinality_cols], Y_train)
X_val_encoded[high_cardinality_cols] = encoder.transform(X_val[high_cardinality_cols])


High cardinality columns: ['building_weekday_hour', 'building_weekday', 'building_month', 'building_hour', 'building_meter']


In [20]:
#HistGradientBoosting modeling
hist_model = HistGradientBoostingClassifier(random_state=42)
hist_model.fit(X_train_encoded, Y_train)

pred_train_hist = hist_model.predict_proba(X_train_encoded)[:,1]
pred_val_hist = hist_model.predict_proba(X_val_encoded)[:,1]

score_train_hist = metrics.roc_auc_score(Y_train, pred_train_hist)
score_val_hist = metrics.roc_auc_score(Y_val, pred_val_hist)

print('Training Accuracy : ', score_train_hist)
print('Validation Accuracy : ', score_val_hist)

Training Accuracy :  0.9987857243820558
Validation Accuracy :  0.9398186588652767


In [ ]:
#Catboost modeling
cat_features = list(X_train_encoded.select_dtypes(include=['category', 'object']).columns)
cat_model = CatBoostClassifier(
    iterations=100, #用iterations = 10 試跑，發現表現就很好了(0.99/0.93)，先用100看看
    random_seed=42,
    cat_features=cat_features
)
cat_model.fit(X_train_encoded, Y_train, silent=True)

pred_train_cat = cat_model.predict_proba(X_train_encoded)[:,1]
pred_val_cat = cat_model.predict_proba(X_val_encoded)[:,1]

score_train_cat = metrics.roc_auc_score(Y_train, pred_train_cat)
score_val_cat = metrics.roc_auc_score(Y_val, pred_val_cat)

print('Training Accuracy : ', score_train_cat)
print('Validation Accuracy : ', score_val_cat)

Training Accuracy :  0.9990767524728446
Validation Accuracy :  0.9308771525236801


In [27]:
#Model ensembling
score_train_ensemble = metrics.roc_auc_score(Y_train, (pred_train_xgb+pred_train_cat+pred_train_lgb+pred_train_hist)/4)
score_val_ensemble = metrics.roc_auc_score(Y_val, (pred_val_xgb+pred_val_cat+pred_val_lgb+pred_val_hist)/4)

print('Training Accuracy : ', score_train_ensemble)
print('Validation Accuracy : ', score_val_ensemble)

Training Accuracy :  0.999753338051897
Validation Accuracy :  0.9414765139675477


In [ ]:
import gc
X_all_encoded = X_all.copy()
X_all_encoded[high_cardinality_cols] = encoder.transform(X_all[high_cardinality_cols])
del X_train, X_train_encoded, X_val, X_val_encoded, Y_train, Y_val
gc.collect()

In [32]:
lgb_model.fit(X_all, Y_all)
print('LightGBM model training complete.')
xgb_model.fit(X_all, Y_all)
print('XGBoost model training complete.')
cat_model.fit(X_all_encoded, Y_all, silent=True)
print('CatBoost model training complete.')
hist_model.fit(X_all_encoded, Y_all)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 642256, number of negative: 9046443
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.455996 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 70350
[LightGBM] [Info] Number of data points in the train set: 9688699, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.066289 -> initscore=-2.645140
[LightGBM] [Info] Start training from score -2.645140
LightGBM model training complete.
XGBoost model training complete.
CatBoost model training complete.


,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide `... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 1.4 Added `""from_dtype""` option... versionchanged:: 1.6 The default value changed from `None` to `""from_dtype""`.",'from_dt

In [39]:
del X_all, X_all_encoded, Y_all
gc.collect()

7

In [41]:
pred_test = (xgb_model.predict_proba(X_test)[:,1] + lgb_model.predict_proba(X_test)[:,1])
X_test[high_cardinality_cols] = encoder.transform(X_test[high_cardinality_cols])
pred_test = (pred_test + cat_model.predict_proba(X_test)[:,1] + hist_model.predict_proba(X_test)[:,1]) / 4 

score_test = metrics.roc_auc_score(test_features['is_bad_meter_reading'], pred_test)
print('Test Accuracy : ', score_test)

Test Accuracy :  0.5244186210889196


In [43]:
res = pd.DataFrame({'anomaly': pred_test})
res.to_pickle('data/output/pred_test0519_0.5244.pkl')